# Minería de Datos · Análisis de Datos de Hurto y Capacidad Policial en Colombia

Este cuaderno realiza la carga, diagnóstico, limpieza, exportación individual e integración de los datasets oficiales de:
1. **Hurto de Vehículos** (Policía Nacional / DIJIN)
2. **Recuperación de Vehículos** (Policía Nacional / DIJIN)
3. **Crecimiento del Parque Automotor** (RUNT / Ministerio de Transporte)
4. **Municipios DANE** (DIVIPOLA)


## 1. Importación de librerías y clases modulares


In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")

from src.hurto import Hurto
from src.recuperacion import Recuperacion
from src.parque_automotor import ParqueAutomotor
from src.integracion import Integracion


## 2. Dataset de Hurto de Vehículos
### 2.1 Carga de los datos


In [2]:
hurto = Hurto("../dataset/HurtoVehiculos.csv")
df_hurto = hurto.cargar()
print("Muestra inicial de registros de hurto:")
df_hurto.head()


Muestra inicial de registros de hurto:


,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,TIPO DELITO,ZONA,CANTIDAD
0,31/07/2026,11,BOGOTA D.C.,11001,BOGOTA D.C.,ARTICULO 239. HURTO MOTOCICLETAS,URBANA,10
1,31/07/2026,44,LA GUAJIRA,44001,RIOHACHA,ARTICULO 239. HURTO AUTOMOTORES,URBANA,1
2,31/07/2026,54,NORTE DE SANTANDER,54001,CUCUTA,ARTICULO 239. HURTO MOTOCICLETAS,URBANA,1
3,31/07/2026,50,META,50001,VILLAVICENCIO,ARTICULO 239. HURTO MOTOCICLETAS,URBANA,4
4,31/07/2026,19,CAUCA,19001,POPAYAN,ARTICULO 239. HURTO MOTOCICLETAS,URBANA,3


### 2.2 Diagnóstico inicial


In [3]:
hurto.diagnosticar()


Filas: 386,071
Columnas: 8

Valores nulos por columna:
FECHA HECHO     0
COD_DEPTO       0
DEPARTAMENTO    0
COD_MUNI        0
MUNICIPIO       0
TIPO DELITO     0
ZONA            0
CANTIDAD        0
dtype: int64

Duplicados exactos: 0


### 2.3 Limpieza de datos (normalización, deduplicación y tipología vehicular)


In [4]:
df_hurto_limpio = hurto.limpiar()
print("Diagnóstico post-limpieza de Hurtos:")
hurto.diagnosticar()


Diagnóstico post-limpieza de Hurtos:
Filas: 386,071
Columnas: 12

Valores nulos por columna:
FECHA HECHO              0
COD_DEPTO                0
DEPARTAMENTO             0
COD_MUNI                 0
MUNICIPIO                0
TIPO DELITO              0
ZONA                     0
CANTIDAD                 0
CODIGO_DANE_MUNICIPIO    0
FECHA                    0
AÑO                      0
TIPO_VEHICULO            0
dtype: int64

Duplicados exactos: 0


### 2.4 Exportación del dataset limpio


In [ ]:
hurto.guardar("../dataset/HurtoVehiculos_limpio.csv")


## 3. Dataset de Recuperación de Vehículos
### 3.1 Carga de los datos


In [ ]:
recuperacion = Recuperacion("../dataset/RecuperacionPoliciaNacional.csv")
df_rec = recuperacion.cargar()
print("Muestra inicial de registros de recuperación:")
df_rec.head()


### 3.2 Diagnóstico inicial


In [ ]:
recuperacion.diagnosticar()


### 3.3 Limpieza de datos (eliminación de duplicados exactos y estandarización DANE)


In [ ]:
df_rec_limpio = recuperacion.limpiar()
print("Diagnóstico post-limpieza de Recuperaciones:")
recuperacion.diagnosticar()


### 3.4 Exportación del dataset limpio


In [ ]:
recuperacion.guardar("../dataset/RecuperacionPoliciaNacional_limpio.csv")


## 4. Dataset de Crecimiento del Parque Automotor (RUNT)
### 4.1 Carga de los datos


In [ ]:
parque = ParqueAutomotor("../dataset/CrecimientoParqueAutomotor.csv")
df_parque = parque.cargar()
print("Muestra inicial de registros del RUNT:")
df_parque.head()


### 4.2 Diagnóstico inicial


In [ ]:
parque.diagnosticar()


### 4.3 Limpieza de datos (filtrado de activos y tratamiento del año 1900)


In [ ]:
df_parque_limpio = parque.limpiar()
print("Diagnóstico post-limpieza de Parque Automotor:")
parque.diagnosticar()


### 4.4 Exportación del dataset limpio


In [ ]:
parque.guardar("../dataset/CrecimientoParqueAutomotor_limpio.csv")


## 5. Integración de los tres datasets
Se consolida la información a nivel municipal y anual, cruzando por código DANE y tipo de vehículo (MOTOCICLETA / AUTOMOTOR).


In [ ]:
integrador = Integracion(
    ruta_hurto="../dataset/HurtoVehiculos_limpio.csv",
    ruta_recuperacion="../dataset/RecuperacionPoliciaNacional_limpio.csv",
    ruta_parque="../dataset/CrecimientoParqueAutomotor_limpio.csv",
    ruta_dane="../dataset/municipios_dane.xlsx"
)
df_integrado = integrador.integrar()
print("Primeros 10 registros del dataset integrado:")
df_integrado.head(10)


### 5.1 Diagnóstico del dataset integrado


In [ ]:
integrador.diagnosticar()


### 5.2 Exportación del dataset integrado a CSV y Excel


In [ ]:
integrador.guardar("../dataset/dataset_integrado.csv", "../dataset/dataset_integrado.xlsx")


## 6. Análisis Exploratorio de Datos (EDA)
### 6.1 Métricas globales de negocio


In [ ]:
total_hurtados = df_integrado["VEHICULOS_HURTADOS"].sum()
total_recuperados = df_integrado["VEHICULOS_RECUPERADOS"].sum()
total_registrados = df_integrado["VEHICULOS_REGISTRADOS"].sum()
tasa_recuperacion_global = (total_recuperados / total_hurtados) * 100 if total_hurtados > 0 else 0

print(f"Total Hurtos Registrados: {int(total_hurtados):,}")
print(f"Total Recuperaciones Policiales: {int(total_recuperados):,}")
print(f"Total Parque Automotor Registrado: {int(total_registrados):,}")
print(f"Tasa Global de Recuperación: {tasa_recuperacion_global:.2f}%")


### 6.2 Distribución de hurtos y recuperaciones por tipo de vehículo


In [ ]:
tipo_vehiculo_resumen = df_integrado.groupby("TIPO_VEHICULO")[["VEHICULOS_HURTADOS", "VEHICULOS_RECUPERADOS", "VEHICULOS_REGISTRADOS"]].sum()
tipo_vehiculo_resumen["TASA_RECUPERACION_%"] = (tipo_vehiculo_resumen["VEHICULOS_RECUPERADOS"] / tipo_vehiculo_resumen["VEHICULOS_HURTADOS"]) * 100
tipo_vehiculo_resumen.round(2)


### 6.3 Top 5 Departamentos con mayor incidencia de hurto


In [ ]:
top_deptos = df_integrado.groupby("DEPARTAMENTO")["VEHICULOS_HURTADOS"].sum().sort_values(ascending=False).head(5)
print("Top 5 Departamentos con más hurtos:")
for depto, cant in top_deptos.items():
    print(f" - {depto}: {int(cant):,} hurtos")


## 7. Visualizaciones de Patrones de Seguridad Vial


In [ ]:
plt.figure(figsize=(8, 4))
tipo_vehiculo_resumen["VEHICULOS_HURTADOS"].plot(kind="bar", color=["#1f77b4", "#ff7f0e"])
plt.title("Hurtos Registrados según Tipo de Vehículo")
plt.ylabel("Cantidad de Hurtos")
plt.xlabel("Tipo de Vehículo")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
evolucion_anual = df_integrado.groupby("AÑO")[["VEHICULOS_HURTADOS", "VEHICULOS_RECUPERADOS"]].sum()
evolucion_anual.plot(marker="o")
plt.title("Evolución Anual de Hurtos vs. Recuperaciones Policiales en Colombia")
plt.ylabel("Cantidad")
plt.xlabel("Año")
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()


## 8. Conclusiones
- La integración multifuente a nivel de código municipal DANE y año permite correlacionar el parque automotor expuesto con los incidentes delictivos y los operativos de recuperación.
- Las motocicletas representan la tipología vehicular con mayor vulnerabilidad delictiva en Colombia.
- La base integrada constituye el insumo verificado para el proceso de transformación, escalamiento y reducción de dimensionalidad mediante PCA.
